In [1]:
!pip install -q transformers sentencepiece x-transformers optuna scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.8 MB/s eta 0:00:00


In [2]:
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.44.2 tokenizers==0.19.1

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 105.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 97.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 80.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 39.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 96.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 15.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 35.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 15.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/19

In [3]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from x_transformers import Decoder

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    hamming_loss
)

import optuna

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", device)

DEVICE: cuda


In [6]:
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

MAX_LEN = 128

EPOCHS = 100

BATCH_SIZE = 64

NUM_LABELS = 6


In [7]:
CSV_PATH = "/kaggle/input/datasets/kmkimmy/sied-thai/SIED-Thai.csv"

df = pd.read_csv(CSV_PATH)

TEXT_COLUMN = "Tweets"

LABEL_COLUMNS = [
    "Happiness",
    "Sadness",
    "Anger",
    "Disgust",
    "Surprise",
    "Fear"
]

print(df.head())

   No.  Labels                                             Tweets  Happiness  \
0    1       0  เป็นได้แค่คนคุย มันทรมานจังเนาะ. ?? . . . . . ...          0   
1    2       0  กุรอคุยมาทั้งวัน!! แต่ได้คำตอบสั้นๆว่า.. ฝันดี...          0   
2    3       0  เมื่อกี้ยังอยู่กับเพื่อน ดื่มไวน์ เม้ามอยเฮฮา ...          0   
3    4       0  บางคนอาจสงสัยการแอดมิดหวอดจิตเวชว่าต่างจากแอดม...          0   
4    5       0  ด่วน! นายกฯ ออกแถลงการณ์ แจงสถานการณ์”ปาบึก”สั...          0   

   Sadness  Anger  Disgust  Surprise  Fear  Sentiment  
0        1      0        0         0     0          2  
1        1      1        0         0     0          2  
2        1      0        0         1     0          2  
3        0      0        0         0     0          0  
4        0      0        0         1     0          0  


In [8]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=SEED,
    shuffle=True
)

print("TRAIN:", len(train_df))
print("VAL:", len(val_df))
print("TEST:", len(test_df))


TRAIN: 1728
VAL: 192
TEST: 480


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

In [10]:
class EmotionDataset(Dataset):

    def __init__(self, df):

        self.texts = df[TEXT_COLUMN].tolist()

        self.labels = df[LABEL_COLUMNS].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        labels = self.labels[idx]

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(labels, dtype=torch.float)
        }
train_dataset = EmotionDataset(train_df)

val_dataset = EmotionDataset(val_df)

test_dataset = EmotionDataset(test_df)

In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [12]:
label_counts = train_df[LABEL_COLUMNS].sum().values

total_samples = len(train_df)

pos_weights = total_samples / (label_counts + 1e-6)

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float
).to(device)

print("POS WEIGHTS:", pos_weights)

POS WEIGHTS: tensor([14.1639,  1.2838,  7.5459,  5.0973, 13.9355,  7.4163], device='cuda:0')


In [13]:
class AttentionPooling(nn.Module):

    def __init__(self, hidden_size):

        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x, mask):

        scores = self.attention(x).squeeze(-1)

        scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)

        weights = torch.softmax(scores, dim=1)

        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        return pooled

In [14]:
class EmotionModel(nn.Module):

    def __init__(
        self,
        depth=1,
        heads=2,
        attn_dropout=0.2,
        ff_dropout=0.2,
        ff_mult=2
    ):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME
        )

        hidden_size = self.encoder.config.hidden_size

        self.decoder = Decoder(
            dim=hidden_size,
            depth=depth,
            heads=heads,
            attn_dropout=attn_dropout,
            ff_dropout=ff_dropout,
            ff_mult=ff_mult
        )

        self.pooling = AttentionPooling(hidden_size)

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(
            hidden_size,
            NUM_LABELS
        )

    def forward(self, input_ids, attention_mask):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        x = outputs.last_hidden_state

        x = self.decoder(x)

        x = self.pooling(x, attention_mask)

        x = self.dropout(x)

        logits = self.fc(x)

        return logits

In [15]:
def optimize_thresholds(y_true, y_probs):

    best_thresholds = []

    for i in range(NUM_LABELS):

        best_thr = 0.5
        best_f1 = 0

        for thr in np.arange(0.1, 0.9, 0.05):

            preds = (y_probs[:, i] >= thr).astype(int)

            score = f1_score(
                y_true[:, i],
                preds,
                zero_division=0
            )

            if score > best_f1:

                best_f1 = score

                best_thr = thr

        best_thresholds.append(best_thr)

    return np.array(best_thresholds)

In [16]:
def evaluate(model, loader, thresholds=None):

    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            logits = model(
                input_ids,
                attention_mask
            )

            probs = torch.sigmoid(logits)

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                labels.cpu().numpy()
            )

    all_probs = np.vstack(all_probs)

    all_labels = np.vstack(all_labels)

    if thresholds is None:

        thresholds = np.array(
            [0.5] * NUM_LABELS
        )

    preds = (
        all_probs >= thresholds
    ).astype(int)

    micro_f1 = f1_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_labels,
        preds,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(
        all_labels,
        preds
    )

    h_loss = hamming_loss(
        all_labels,
        preds
    )

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "accuracy": accuracy,
        "hamming_loss": h_loss,
        "labels": all_labels,
        "preds": preds,
        "probs": all_probs
    }


In [17]:
def train_model(
    model,
    train_loader,
    val_loader,
    lr,
    weight_decay
):

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weights
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = len(train_loader) * EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(
            0.1 * total_steps
        ),
        num_training_steps=total_steps
    )

    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        loop = tqdm(train_loader)

        for batch in loop:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast():

                logits = model(
                    input_ids,
                    attention_mask
                )

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(loss).backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(optimizer)

            scaler.update()

            scheduler.step()

            total_loss += loss.item()

            loop.set_description(
                f"Epoch {epoch+1}/{EPOCHS}"
            )

            loop.set_postfix(
                loss=loss.item()
            )

        val_result = evaluate(
            model,
            val_loader
        )

        print("\n======================")
        print(f"Epoch {epoch+1}")
        print("======================")

        print(
            f"VAL ACCURACY   : {val_result['accuracy']:.4f}"
        )

        print(
            f"VAL MICRO F1   : {val_result['micro_f1']:.4f}"
        )

        print(
            f"VAL MACRO F1   : {val_result['macro_f1']:.4f}"
        )

        print(
            f"VAL WEIGHTED F1: {val_result['weighted_f1']:.4f}"
        )

    return model


In [18]:
def objective(trial):

    depth = trial.suggest_int(
        "depth",
        1,
        2
    )

    heads = trial.suggest_categorical(
        "heads",
        [2, 4]
    )

    attn_dropout = trial.suggest_float(
        "attn_dropout",
        0.2,
        0.5
    )

    ff_dropout = trial.suggest_float(
        "ff_dropout",
        0.2,
        0.5
    )

    ff_mult = trial.suggest_int(
        "ff_mult",
        2,
        4
    )

    lr = trial.suggest_float(
        "lr",
        1e-5,
        2e-4,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-4,
        1e-2,
        log=True
    )

    model = EmotionModel(
        depth=depth,
        heads=heads,
        attn_dropout=attn_dropout,
        ff_dropout=ff_dropout,
        ff_mult=ff_mult
    ).to(device)

    model = train_model(
        model,
        train_loader,
        val_loader,
        lr,
        weight_decay
    )

    val_result = evaluate(
        model,
        val_loader
    )

    best_thresholds = optimize_thresholds(
        val_result["labels"],
        val_result["probs"]
    )

    val_result = evaluate(
        model,
        val_loader,
        best_thresholds
    )

    return val_result["macro_f1"]

In [19]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=10
)

[I 2026-05-18 04:53:21,330] A new study created in memory with name: no-name-8505490d-fdd5-4a4a-baa8-5752e9d6f77c


model.safetensors:   0%|          | 0.00/423M [00:00<?, ?B/s]

Epoch 1/100: 100%|██████████| 27/27 [00:21<00:00,  1.26it/s, loss=1.25]



Epoch 1
VAL ACCURACY   : 0.0104
VAL MICRO F1   : 0.4735
VAL MACRO F1   : 0.3234
VAL WEIGHTED F1: 0.6170


Epoch 2/100: 100%|██████████| 27/27 [00:20<00:00,  1.29it/s, loss=1.24]


...

Epoch 99/100: 100%|██████████| 27/27 [00:20<00:00,  1.30it/s, loss=8.15e-5]



Epoch 99
VAL ACCURACY   : 0.4844
VAL MICRO F1   : 0.7279
VAL MACRO F1   : 0.5441
VAL WEIGHTED F1: 0.7309


Epoch 100/100: 100%|██████████| 27/27 [00:20<00:00,  1.30it/s, loss=6.4e-5]



Epoch 100
VAL ACCURACY   : 0.4844
VAL MICRO F1   : 0.7279
VAL MACRO F1   : 0.5441
VAL WEIGHTED F1: 0.7309


[I 2026-05-18 10:59:35,597] Trial 9 finished with value: 0.5569051858933747 and parameters: {'depth': 1, 'heads': 2, 'attn_dropout': 0.2300977940140689, 'ff_dropout': 0.31452487977320326, 'ff_mult': 4, 'lr': 5.518992543371784e-05, 'weight_decay': 0.0016459486069056626}. Best is trial 2 with value: 0.6133297933891902.


In [20]:
print("\n======================")
print("BEST PARAMETERS")
print("======================")

print(study.best_params)

best_params = study.best_params


BEST PARAMETERS
{'depth': 2, 'heads': 4, 'attn_dropout': 0.42780092819742693, 'ff_dropout': 0.45659903330825574, 'ff_mult': 3, 'lr': 4.4638200794376354e-05, 'weight_decay': 0.00014211620758672024}


In [21]:
final_model = EmotionModel(
    depth=best_params["depth"],
    heads=best_params["heads"],
    attn_dropout=best_params["attn_dropout"],
    ff_dropout=best_params["ff_dropout"],
    ff_mult=best_params["ff_mult"]
).to(device)

In [22]:
final_model = train_model(
    final_model,
    train_loader,
    val_loader,
    best_params["lr"],
    best_params["weight_decay"]
)

Epoch 1/100: 100%|██████████| 27/27 [00:21<00:00,  1.26it/s, loss=1.39]



Epoch 1
VAL ACCURACY   : 0.0052
VAL MICRO F1   : 0.3582
VAL MACRO F1   : 0.2712
VAL WEIGHTED F1: 0.5260


Epoch 2/100: 100%|██████████| 27/27 [00:21<00:00,  1.25it/s, loss=1.25]



Epoch 2
VAL ACCURACY   : 0.0000
VAL MICRO F1   : 0.4541
VAL MACRO F1   : 0.3374
VAL WEIGHTED F1: 0.6248


Epoch 3/100: 100%|██████████| 27/27 [00:21<00:00,  1.25it/s, loss=1.02]


...


Epoch 98
VAL ACCURACY   : 0.4896
VAL MICRO F1   : 0.7482
VAL MACRO F1   : 0.5609
VAL WEIGHTED F1: 0.7493


Epoch 99/100: 100%|██████████| 27/27 [00:21<00:00,  1.25it/s, loss=0.00154]



Epoch 99
VAL ACCURACY   : 0.4948
VAL MICRO F1   : 0.7523
VAL MACRO F1   : 0.5645
VAL WEIGHTED F1: 0.7519


Epoch 100/100: 100%|██████████| 27/27 [00:21<00:00,  1.25it/s, loss=0.00577]



Epoch 100
VAL ACCURACY   : 0.4948
VAL MICRO F1   : 0.7486
VAL MACRO F1   : 0.5623
VAL WEIGHTED F1: 0.7492


In [23]:
val_result = evaluate(
    final_model,
    val_loader
)

best_thresholds = optimize_thresholds(
    val_result["labels"],
    val_result["probs"]
)

print("\n======================")
print("BEST THRESHOLDS")
print("======================")

for label, thr in zip(
    LABEL_COLUMNS,
    best_thresholds
):
    print(f"{label}: {thr:.2f}")


BEST THRESHOLDS
Happiness: 0.50
Sadness: 0.35
Anger: 0.75
Disgust: 0.60
Surprise: 0.85
Fear: 0.30


In [24]:
test_result = evaluate(
    final_model,
    test_loader,
    best_thresholds
)

print("\n======================")
print("FINAL RESULT")
print("======================")

print(
    f"Accuracy     : {test_result['accuracy']:.4f}"
)

print(
    f"Micro F1     : {test_result['micro_f1']:.4f}"
)

print(
    f"Macro F1     : {test_result['macro_f1']:.4f}"
)

print(
    f"Weighted F1  : {test_result['weighted_f1']:.4f}"
)

print(
    f"Hamming Loss : {test_result['hamming_loss']:.4f}"
)


FINAL RESULT
Accuracy     : 0.5354
Micro F1     : 0.7598
Macro F1     : 0.5768
Weighted F1  : 0.7499
Hamming Loss : 0.1062


In [25]:
print("\n======================")
print("CLASSIFICATION REPORT")
print("======================")

print(
    classification_report(
        test_result["labels"],
        test_result["preds"],
        target_names=LABEL_COLUMNS,
        zero_division=0
    )
)


CLASSIFICATION REPORT
              precision    recall  f1-score   support

   Happiness       0.79      0.63      0.70        35
     Sadness       0.89      0.95      0.92       387
       Anger       0.57      0.32      0.41        53
     Disgust       0.46      0.51      0.48        85
    Surprise       0.75      0.43      0.55        28
        Fear       0.40      0.41      0.40        51

   micro avg       0.76      0.76      0.76       639
   macro avg       0.64      0.54      0.58       639
weighted avg       0.75      0.76      0.75       639
 samples avg       0.76      0.77      0.74       639



In [26]:
SAVE_PATH = "/content/final_emotion_model.pt"

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "thresholds": best_thresholds,
        "best_params": best_params,
        "labels": LABEL_COLUMNS
    },
    SAVE_PATH
)

print("\nMODEL SAVED:", SAVE_PATH)


MODEL SAVED: /content/final_emotion_model.pt
